In [ ]:
# ============================================================
# 电商购买行为预测 (Revenue Prediction) - LightGBM 二分类
# 说明: 代码按顺序分块编写,方便在 Jupyter Notebook 中逐 Cell 运行调试
# ============================================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 基础数据处理库
from scipy.stats import randint as sp_randint, uniform
# 模型相关库
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
import xgboost as xgb
import lightgbm as lgb
# 数据预处理与特征工程
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold
# 模型评估与交叉验证
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score ,accuracy_score, precision_score, recall_score,roc_auc_score, precision_recall_curve

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ---------- 0. 读取数据 ----------
train = pd.read_csv('ecommerce_train.csv')
test  = pd.read_csv('ecommerce_test.csv')

# print('train shape:', train.shape)
# print('test  shape:', test.shape)
# print('\n目标分布:\n', train['Revenue'].value_counts(normalize=True))
# print('\n缺失值(train):\n', train.isna().sum())
# train.head()

In [3]:
# ============================================================
# 1. 基础预处理: 数据类型 / 缺失值 / 字符串空白异常值
# ============================================================
# 合并方便统一处理(最后再拆分)
test['Revenue'] = np.nan
full = pd.concat([train, test], axis=0, ignore_index=True)
print('合并后 shape:', full.shape)

# --- 1.1 字符串型类别列: 去除空白,空白串视作缺失 ---
cat_obj_cols = ['Month', 'VisitorType', 'User_Horoscope']
for c in cat_obj_cols:
    full[c] = full[c].astype(str).str.strip()
    full.loc[full[c].isin(['', 'nan', 'NaN', 'None', 'NULL']), c] = np.nan
    full[c] = full[c].astype('category')

# --- 1.2 布尔列转 0/1 ---
full['Weekend']  = full['Weekend'].astype('boolean')
full['Weekend']  = full['Weekend'].fillna(False).astype(int)
# Revenue 训练集为 bool,测试集为 NaN;先转类别留待后面拆分
full['Revenue']  = full['Revenue'].astype('boolean')

# --- 1.3 数值列中的缺失值 ---
# 以下列在训练/测试中均有约30%缺失: Admin_Duration / BounceRates / PageValues / VisitorType
# 缺失近似随机(MCAR),用中位数填充,并新增"是否缺失"指示特征(缺失本身可能含信息)
num_cols_with_na = ['Admin_Duration', 'BounceRates', 'PageValues']
for c in num_cols_with_na:
    full[c + '_isna'] = full[c].isna().astype(int)        # 缺失指示特征
    med = full.loc[full['Revenue'].notna(), c].median()    # 仅用训练集中位数,防泄漏
    full[c] = full[c].fillna(med)

# VisitorType 缺失填为新类别
full['VisitorType'] = full['VisitorType'].cat.add_categories(['Unknown']).fillna('Unknown')

# --- 1.4 类别型编码: category 类型直接给 LightGBM(LGBM 原生支持类别) ---
# OperatingSystems/Browser/Region/TrafficType 也是类别型(代码),转为 category
for c in ['OperatingSystems', 'Browser', 'Region', 'TrafficType']:
    full[c] = full[c].astype('category')

full.head()

合并后 shape: (350000, 20)


,ID,Administrative,Admin_Duration,Informational,Info_Duration,ProductRelated,Product_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,User_Horoscope,Revenue,Admin_Duration_isna,BounceRates_isna,PageValues_isna
0,1,4,14.776893,1,64.392933,32,559.532750,0.122293,0.202730,4.457813,0.2,Jul,7,8,2,7,New_Visitor,0,Gemini,True,0,0,0
1,2,1,55.519455,3,39.849162,19,1134.915949,0.066891,0.132891,15.477588,0.0,Mar,1,1,4,1,Unknown,0,Virgo,False,0,1,0
2,3,3,164.841695,1,3.218469,29,1107.957351,0.195989,0.051108,19.314592,0.6,Mar,5,7,5,1,Other,0,Leo,False,0,0,0
3,4,3,62.944306,1,10.849725,26,1618.120233,0.066891,0.233461,0.448885,0.2,Sep,7,5,5,6,Other,1,Leo,True,0,1,0
4,5,1,145.274641,2,90.188903,40,292.161045,0.020284,0.192949,11.219882,0.2,Oct,3,9,7,2,Returning_Visitor,0,Aries,False,0,0,0


In [4]:
# ============================================================
# 2. 异常值处理
# ============================================================
# 数值型连续特征存在长尾/极大值(如 Product_Duration 最大 ~7800, PageValues ~124)
# 采用训练集分位数 clip(上界=99.5分位),缩尾降低极端值对树模型分裂的影响
num_cont = ['Admin_Duration', 'Info_Duration', 'Product_Duration',
            'BounceRates', 'ExitRates', 'PageValues']

clip_bounds = {}
for c in num_cont:
    q_lo = full.loc[full['Revenue'].notna(), c].quantile(0.005)
    q_hi = full.loc[full['Revenue'].notna(), c].quantile(0.995)
    clip_bounds[c] = (q_lo, q_hi)
    full[c] = full[c].clip(q_lo, q_hi)

print('缩尾分位界(99.5%):')
for c, (lo, hi) in clip_bounds.items():
    print(f'  {c}: [{lo:.4f}, {hi:.4f}]')

# ============================================================
# 3. 特征工程 (视情况新增特征)
# ============================================================
# 数据中真实信号集中在 Product_Duration / PageValues(相关性最高),
# 其余多为噪声(User_Horoscope 等). 这里构造若干组合/比率/聚合特征增强信号.

# --- 3.1 总停留时长 / 总页面数 ---
full['Total_Duration'] = (full['Admin_Duration'] + full['Info_Duration']
                          + full['Product_Duration'])
full['Total_Pages'] = (full['Administrative'] + full['Informational']
                       + full['ProductRelated'])

# --- 3.2 平均每页停留时长(用户粘性) ---
full['Avg_Duration_per_Page'] = full['Total_Duration'] / (full['Total_Pages'] + 1)
full['Avg_Product_Duration']  = full['Product_Duration'] / (full['ProductRelated'] + 1)
full['Avg_Admin_Duration']    = full['Admin_Duration']   / (full['Administrative'] + 1)
full['Avg_Info_Duration']     = full['Info_Duration']    / (full['Informational'] + 1)

# --- 3.3 各类页面占比(用户行为结构) ---
full['Product_Pages_Ratio'] = full['ProductRelated'] / (full['Total_Pages'] + 1)
full['Admin_Pages_Ratio']   = full['Administrative'] / (full['Total_Pages'] + 1)
full['Info_Pages_Ratio']    = full['Informational']  / (full['Total_Pages'] + 1)

# --- 3.4 各类页面时长占比 ---
full['Product_Duration_Ratio'] = full['Product_Duration'] / (full['Total_Duration'] + 1)
full['Admin_Duration_Ratio']   = full['Admin_Duration']   / (full['Total_Duration'] + 1)

# --- 3.5 跳出/离开相关组合 ---
full['Bounce_Exit_Mean'] = (full['BounceRates'] + full['ExitRates']) / 2
full['High_Bounce']      = (full['BounceRates'] > 0.2).astype(int)
full['Low_Exit']         = (full['ExitRates'] < 0.05).astype(int)

# --- 3.6 PageValues 相关 ---
full['PageValues_Log']  = np.log1p(full['PageValues'])
full['High_PageValues'] = (full['PageValues'] > full['PageValues'].quantile(0.75)).astype(int)

# --- 3.7 特节庆/周末交互 ---
full['SpecialDay_High'] = (full['SpecialDay'] > 0.5).astype(int)

# --- 3.8 月份顺序编码(月份本身含时间序) ---
month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'June':6,'Jul':7,
             'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
full['Month_num'] = full['Month'].map(month_map).astype('float')

print('新增特征后 shape:', full.shape)
full.head()

缩尾分位界(99.5%):
  Admin_Duration: [0.5671, 395.2463]
  Info_Duration: [0.1951, 211.8427]
  Product_Duration: [51.0431, 3729.6412]
  BounceRates: [0.0007, 0.3911]
  ExitRates: [0.0099, 0.5077]
  PageValues: [0.0741, 49.1516]
新增特征后 shape: (350000, 41)


,ID,Administrative,Admin_Duration,Informational,Info_Duration,ProductRelated,Product_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,User_Horoscope,Revenue,Admin_Duration_isna,BounceRates_isna,PageValues_isna,Total_Duration,Total_Pages,Avg_Duration_per_Page,Avg_Product_Duration,Avg_Admin_Duration,Avg_Info_Duration,Product_Pages_Ratio,Admin_Pages_Ratio,Info_Pages_Ratio,Product_Duration_Ratio,Admin_Duration_Ratio,Bounce_Exit_Mean,High_Bounce,Low_Exit,PageValues_Log,High_PageValues,SpecialDay_High,Month_num
0,1,4,14.776893,1,64.392933,32,559.532750,0.122293,0.202730,4.457813,0.2,Jul,7,8,2,7,New_Visitor,0,Gemini,True,0,0,0,638.702576,37,16.807963,16.955538,2.955379,32.196466,0.842105,0.105263,0.026316,0.874676,0.023100,0.162511,0,0,1.697048,0,0,7.0
1,2,1,55.519455,3,39.849162,19,1134.915949,0.066891,0.132891,15.477588,0.0,Mar,1,1,4,1,Unknown,0,Virgo,False,0,1,0,1230.284566,23,51.261857,56.745797,27.759728,9.962290,0.791667,0.041667,0.125000,0.921733,0.045091,0.099891,0,0,2.802001,1,0,3.0
2,3,3,164.841695,1,3.218469,29,1107.957351,0.195989,0.051108,19.314592,0.6,Mar,5,7,5,1,Other,0,Leo,False,0,0,0,1276.017515,33,37.529927,36.931912,41.210424,1.609234,0.852941,0.088235,0.029412,0.867613,0.129083,0.123548,0,0,3.011339,1,1,3.0
3,4,3,62.944306,1,10.849725,26,1618.120233,0.066891,0.233461,0.448885,0.2,Sep,7,5,5,6,Other,1,Leo,True,0,1,0,1691.914264,30,54.577879,59.930379,15.736076,5.424862,0.838710,0.096774,0.032258,0.955819,0.037181,0.150176,0,0,0.370794,0,0,9.0
4,5,1,145.274641,2,90.188903,40,292.161045,0.020284,0.192949,11.219882,0.2,Oct,3,9,7,2,Returning_Visitor,0,Aries,False,0,0,0,527.624589,43,11.991468,7.125879,72.637320,30.062968,0.909091,0.022727,0.045455,0.552682,0.274816,0.106617,0,0,2.503064,1,0,10.0


In [5]:
# ============================================================
# 4. 拆分训练/测试, 准备特征矩阵
# ============================================================
train_df = full[full['Revenue'].notna()].copy()
test_df  = full[full['Revenue'].isna()].copy()

# 目标变量
y = train_df['Revenue'].astype(int).values

# 噪声特征评估: User_Horoscope 各星座购买率几乎一致(~13.5%),判为干扰特征,剔除
# ID 仅为流水号,无预测意义,剔除
drop_cols = ['ID', 'Revenue', 'User_Horoscope']
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

# category 列名(LightGBM 原生类别特征)
cat_cols = [c for c in feature_cols
            if str(X[c].dtype) == 'category']
print('特征数:', len(feature_cols))
print('类别特征:', cat_cols)

特征数: 38
类别特征: ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']


In [ ]:
# 4.5 类别型特征编码
def encode_features(X_train, X_test, cat_cols):
    """对类别型特征进行编码"""
    encoders = {}
    X_train = X_train.copy()
    X_test  = X_test.copy()

    for col in cat_cols:
        # print(col)
        le = LabelEncoder()
        # 合并训练集和测试集的唯一值进行编码
        all_vals = set(X_train[col].dropna().unique()) | set(X_test[col].dropna().unique())
        le.fit(list(all_vals))

        X_train[col] = le.transform(X_train[col].astype(str))
        X_test[col]  = le.transform(X_test[col].astype(str))
        encoders[col] = le
        # print(f"  {col}: {len(le.classes_)} 个类别 -> 编码为 0~{len(le.classes_)-1}")

    return X_train, X_test, encoders


# 4.6 识别特征类型
cat_features = X.select_dtypes(include='category').columns.tolist()
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
print(f"\n  类别型特征 ({len(cat_features)}): {cat_features}")
print(f"  数值型特征 ({len(num_features)}): {num_features}")

print("\n--- 类别型特征编码 ---")
X, X_test, encoders = encode_features(X, X_test, cat_features)

In [ ]:
# ============================================================
# 5. 多测试几个模型
# ============================================================
models = {
    'XGBoost': xgb.XGBClassifier(
        objective='binary:logistic',  # 多分类是 'multi:softprob'，二分类改为 'binary:logistic'
        # 移除 num_class，二分类不需要该参数
        max_depth=4,
        learning_rate=0.03,
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1,  # XGBoost不支持class_weight，用scale_pos_weight处理不平衡(可设为 负样本数/正样本数)
        random_state=42,
        verbosity=0,
        eval_metric='logloss'  # 二分类评估指标
    ),
    'RF': RandomForestClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced_subsample',  # 无需修改，自动适配二分类
        max_features='sqrt',
        oob_score=True,
        random_state=42,
        n_jobs=-1
    ),
    'LGBM': lgb.LGBMClassifier(
        boosting_type='gbdt',
        num_leaves=31,
        learning_rate=0.05,
        n_estimators=150,
        objective='binary',  # 多分类是 'multiclass'，二分类改为 'binary'
        # 移除 num_class，二分类不需要该参数
        is_unbalance=True,  # 用 is_unbalance 替代 class_weight 处理类别不平衡
        random_state=42,
        importance_type='gain',
        verbose=-1,
        n_jobs=-1
    ),
    'GB': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        subsample=0.8,
        max_features='sqrt',
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42
        # GradientBoosting 自动适配二分类，默认 loss='log_loss'
    ),
    'ExtraTrees': ExtraTreesClassifier(
        n_estimators=200,
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced_subsample',  # 自动适配二分类
        max_features='sqrt',
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
# ============================================================
# 5.5 模型结果
# ============================================================
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.metrics import accuracy_score

# 交叉验证
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = {
    f'{name}_{metric}': [] 
    for name in models.keys() 
    for metric in ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
}

for train_idx, val_idx in skf.split(X, y):
    x_cv_train, x_cv_val = X.values[train_idx], X.values[val_idx]
    y_cv_train, y_cv_val = y[train_idx], y[val_idx]
    
    for name, model in models.items():
        # 重新创建模型实例，避免 fold 间状态泄露
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(x_cv_train, y_cv_train)
        y_pred = model_clone.predict(x_cv_val)
        # 获取预测概率用于计算 AUC
        if hasattr(model_clone, 'predict_proba'):
            y_proba = model_clone.predict_proba(x_cv_val)[:, 1]
        else:
            y_proba = model_clone.decision_function(x_cv_val)
        
        # 二分类指标
        fold_scores[f'{name}_Accuracy'].append(accuracy_score(y_cv_val, y_pred))
        fold_scores[f'{name}_Precision'].append(precision_score(y_cv_val, y_pred))
        fold_scores[f'{name}_Recall'].append(recall_score(y_cv_val, y_pred))
        fold_scores[f'{name}_F1'].append(f1_score(y_cv_val, y_pred))
        fold_scores[f'{name}_AUC'].append(roc_auc_score(y_cv_val, y_proba))

# 结果输出
print("5折交叉验证结果（二分类）：")
print(f"{'模型':<12}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}{'AUC':<12}")
print('-' * 72)

results = []
for name in models.keys():
    acc = np.mean(fold_scores[f'{name}_Accuracy'])
    prec = np.mean(fold_scores[f'{name}_Precision'])
    rec = np.mean(fold_scores[f'{name}_Recall'])
    f1 = np.mean(fold_scores[f'{name}_F1'])
    auc = np.mean(fold_scores[f'{name}_AUC'])
    results.append({
        '模型': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'AUC': round(auc, 4)
    })
    print(f"{name:<12}{acc:<12.4f}{prec:<12.4f}{rec:<12.4f}{f1:<12.4f}{auc:<12.4f}")

5折交叉验证结果（二分类）：
模型          Accuracy    Precision   Recall      F1          AUC         
------------------------------------------------------------------------
XGBoost     0.8646      0.5677      0.0052      0.0103      0.6368      
RF          0.6478      0.1993      0.5297      0.2897      0.6355      
LGBM        0.6536      0.2007      0.5215      0.2898      0.6348      
GB          0.8646      0.5810      0.0051      0.0101      0.6370      
ExtraTrees  0.6503      0.2004      0.5285      0.2906      0.6359      


In [13]:
# ============================================================
# 6. 用表现最好的模型lightGBM，样本不平衡说明 + 超参数调参 (Optuna 贝叶斯寻优)
# ============================================================
# 正样本率 ~13.5% (不平衡). 采用样本权重法: 正类按反频率加权, 不破坏原始分布.
# 不做过采样(SMOTE)避免放大噪声; 通过门坎优化调 F1.
import lightgbm as lgb
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix)
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# ---------- 6.1 Optuna 超参数寻优 ----------
# 目标: 5折 OOF 在"最优门坎"下的 F1, 直接对齐评分指标 (Revenue=TRUE 的 F1).
# 每个 trial 内部用样本权重处理不平衡, early_stopping 加速.
import os
N_TRIALS    = int(os.environ.get('N_TRIALS', '30'))  # 可调大(如50)追求更优; 调小(如10)加速测试
TUNE_N_EST  = 1000  # 调参阶段单折最大轮数(配合早停, 实际多数会提前停)
ES_ROUNDS   = 50    # 早停轮数
print(f'[调参配置] N_TRIALS={N_TRIALS}, TUNE_N_EST={TUNE_N_EST}, ES_ROUNDS={ES_ROUNDS}')

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'n_jobs': -1,
        'random_state': SEED,
        'learning_rate':     trial.suggest_float('learning_rate', 5e-3, 0.3, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 255),
        'max_depth':         trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 0.5),
    }
    oof = np.zeros(len(y))
    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        sw = np.where(y_tr == 1, (y_tr == 0).sum() / (y_tr == 1).sum(), 1.0)
        m = lgb.LGBMClassifier(n_estimators=TUNE_N_EST, **params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)],
              sample_weight=sw)
        oof[va_idx] = m.predict_proba(X_va)[:, 1]
    # 在 OOF 上搜索最优门坎, 返回其 F1 (对齐评分)
    best_f1 = 0.0
    for thr in np.arange(0.05, 0.75, 0.01):
        f1 = f1_score(y, (oof >= thr).astype(int))
        if f1 > best_f1:
            best_f1 = f1
    return best_f1

study = optuna.create_study(direction='maximize', study_name='lgb_f1_tune')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
print(f'\n调参完成: 最优 OOF F1 = {study.best_value:.4f}')
print('最优超参数:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

[调参配置] N_TRIALS=30, TUNE_N_EST=1000, ES_ROUNDS=50


  0%|          | 0/30 [00:00<?, ?it/s]


调参完成: 最优 OOF F1 = 0.2930
最优超参数:
  learning_rate: 0.REDACTED-ID
  num_leaves: 122
  max_depth: 4
  min_child_samples: 184
  feature_fraction: 0.8966161314190464
  bagging_fraction: 0.7655854160112691
  bagging_freq: 7
  reg_alpha: 3.8211208941497334
  reg_lambda: 1.319737872320878
  min_split_gain: 0.3306419084221876


In [15]:
# ---------- 5.2 用最优超参组装最终参数 ----------
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'n_jobs': -1,
    'random_state': SEED,
    'verbose': -1,
    'n_estimators': 2000,
    **best_params,
}
print('\n最终 LightGBM 参数:')
print(lgb_params)


最终 LightGBM 参数:
{'objective': 'binary', 'metric': 'binary_logloss', 'n_jobs': -1, 'random_state': 42, 'verbose': -1, 'n_estimators': 2000, 'learning_rate': 0.REDACTED-ID, 'num_leaves': 122, 'max_depth': 4, 'min_child_samples': 184, 'feature_fraction': 0.8966161314190464, 'bagging_fraction': 0.7655854160112691, 'bagging_freq': 7, 'reg_alpha': 3.8211208941497334, 'reg_lambda': 1.319737872320878, 'min_split_gain': 0.3306419084221876}


In [7]:
# ============================================================
# 6. 5折交叉验证 (输出每折 OOF 概率, 用于门坎优化)
# ============================================================
oof_proba = np.zeros(len(train_df))
test_proba = np.zeros(len(test_df))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
        # 样本不平衡: 用 balanced 自动加权
        sample_weight=np.where(y_tr == 1,
                               (y_tr == 0).sum() / (y_tr == 1).sum(), 1.0),
    )

    oof_proba[va_idx] = model.predict_proba(X_va)[:, 1]
    test_proba += model.predict_proba(X_test)[:, 1] / N_FOLDS

    fold_f1 = f1_score(y_va, (oof_proba[va_idx] >= 0.5).astype(int))
    fold_auc = roc_auc_score(y_va, oof_proba[va_idx])
    print(f'Fold {fold+1}: F1@0.5={fold_f1:.4f}  AUC={fold_auc:.4f}  '
          f'best_iter={model.best_iteration_}')

# 整体 OOF 指标
oof_auc = roc_auc_score(y, oof_proba)
print(f'\nOOF AUC = {oof_auc:.4f}')

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1116]	valid_0's binary_logloss: 0.653299
Fold 1: F1@0.5=0.2875  AUC=0.6350  best_iter=1116
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1095]	valid_0's binary_logloss: 0.657071
Fold 2: F1@0.5=0.2885  AUC=0.6313  best_iter=1095
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1482]	valid_0's binary_logloss: 0.654419
Fold 3: F1@0.5=0.2948  AUC=0.6378  best_iter=1482
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1074]	valid_0's binary_logloss: 0.655915
Fold 4: F1@0.5=0.2915  AUC=0.6358  best_iter=1074
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1992]	valid_0's binary_logloss: 0.652617
Fold 5: F1@0.5=0.2939  AUC=0.6386  best_iter=1992

OOF AUC = 0.6357


In [8]:
# ============================================================
# 7. 门坎值优化 (针对 F1 优化阈值)
# ============================================================
# 评分以 Revenue=TRUE 的 F1 为准, 默认0.5不一定最优.
# 在 OOF 概率上搜索使 F1 最大的阈值.
best_thr, best_f1 = 0.5, 0.0
for thr in np.arange(0.05, 0.95, 0.005):
    f1 = f1_score(y, (oof_proba >= thr).astype(int))
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

print(f'最优门坎 best_thr = {best_thr:.3f}, OOF F1 = {best_f1:.4f}')

# 对比默认0.5
f1_default = f1_score(y, (oof_proba >= 0.5).astype(int))
print(f'默认门坎 0.5   的 OOF F1 = {f1_default:.4f}')


最优门坎 best_thr = 0.515, OOF F1 = 0.2919
默认门坎 0.5   的 OOF F1 = 0.2913


In [9]:
# ============================================================
# 8. 全量训练 (用全部训练数据 + 最优超参/早停近似轮数)
# ============================================================
# 用 5折 CV 中各折 best_iteration 的均值作为全量训练的轮数,
# 以充分利用全部数据; 类别不平衡同样用样本权重处理.
avg_best_iter = int(np.mean([
    lgb.LGBMClassifier(**lgb_params).fit(
        X.iloc[tr_idx], y[tr_idx],
        eval_set=[(X.iloc[va_idx], y[va_idx])],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
        sample_weight=np.where(y[tr_idx] == 1,
                               (y[tr_idx] == 0).sum() / (y[tr_idx] == 1).sum(), 1.0),
    ).best_iteration_ or 1000
    for tr_idx, va_idx in skf.split(X, y)
]))
print('各折平均 best_iteration =', avg_best_iter)

final_params = lgb_params.copy()
final_params['n_estimators'] = avg_best_iter

final_model = lgb.LGBMClassifier(**final_params)
final_model.fit(
    X, y,
    sample_weight=np.where(y == 1, (y == 0).sum() / (y == 1).sum(), 1.0),
)
print('全量训练完成.')


Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1116]	valid_0's binary_logloss: 0.653299
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1095]	valid_0's binary_logloss: 0.657071
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1482]	valid_0's binary_logloss: 0.654419
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1074]	valid_0's binary_logloss: 0.655915
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1992]	valid_0's binary_logloss: 0.652617
各折平均 best_iteration = 1351
全量训练完成.


In [10]:
# ============================================================
# 9. 模型评估 (基于 OOF 与全量模型)
# ============================================================
oof_pred = (oof_proba >= best_thr).astype(int)

print('===== OOF 验证集评估 (门坎=%.3f) =====' % best_thr)
print('F1        :', f1_score(y, oof_pred))
print('Precision :', precision_score(y, oof_pred))
print('Recall    :', recall_score(y, oof_pred))
print('AUC       :', roc_auc_score(y, oof_proba))
print('混淆矩阵:\n', confusion_matrix(y, oof_pred))

# 特征重要性 Top20
imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.booster_.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)
print('\n特征重要性 Top20:')
print(imp.head(20).to_string(index=False))

===== OOF 验证集评估 (门坎=0.515) =====
F1        : 0.29191525194829954
Precision : 0.20897169049487607
Recall    : 0.484034986037199
AUC       : 0.6356663974563866
混淆矩阵:
 [[172494  69548]
 [ 19585  18373]]

特征重要性 Top20:
               feature    importance
      Product_Duration 572037.829952
            PageValues 317401.768335
        Total_Duration 204649.915604
           TrafficType 127856.451186
               Browser  58038.717761
  Avg_Product_Duration  51370.109937
                 Month  36164.752744
                Region  24717.268999
 Avg_Duration_per_Page  24053.800236
  Admin_Duration_Ratio  21342.107592
Product_Duration_Ratio  20116.018685
           BounceRates  19520.007018
   Product_Pages_Ratio  18531.735906
      Bounce_Exit_Mean  17865.524454
     Avg_Info_Duration  17524.271831
             ExitRates  17488.475143
      OperatingSystems  16121.882859
         Info_Duration  14163.301846
    Avg_Admin_Duration  14066.700298
        Admin_Duration  12710.799885


In [ ]:


# ============================================================
# 10. 应用测试数据生成预测结果
# ============================================================
# 测试集概率: 取 5折平均 test_proba 与全量模型预测的融合(更稳健)
test_proba_final = 0.5 * test_proba + 0.5 * final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba_final >= best_thr).astype(int)

# 输出 CSV: id, Predicted_Results (TRUE / FALSE)
submission = pd.DataFrame({
    'id': test_df['ID'].values,
    'Predicted_Results': np.where(test_pred == 1, 'TRUE', 'FALSE'),
})
submission.to_csv('submission.csv', index=False)

print('\n预测结果分布:')
print(submission['Predicted_Results'].value_counts())
print('\n已保存 submission.csv, 示例:')
print(submission.head(10).to_string(index=False))
